# 04 — Machine Learning

## 1. Objetivo

Evaluar si modelos de Machine Learning basados en árboles aportan capacidad
predictiva adicional frente a los modelos estadísticos desarrollados
previamente.

Se utilizarán principalmente dos familias:

- Random Forest, como representante de métodos de *bagging*;
- XGBoost, como representante de métodos de *gradient boosting*.

Los modelos se evaluarán sobre exactamente las mismas particiones utilizadas
por los GLM y mediante el mismo protocolo:

- desempeño out-of-sample;
- calibración;
- métricas específicas según el target;
- recuperación de las cantidades *oracle*;
- costo computacional.

El objetivo no es realizar un benchmark exhaustivo de algoritmos, sino estudiar
si el incremento de flexibilidad permite recuperar mejor la estructura
subyacente del riesgo.

## 2. Imports y configuración

In [3]:
from pathlib import Path
from time import perf_counter
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_poisson_deviance,
    mean_gamma_deviance,
    mean_tweedie_deviance,
)

from xgboost import XGBRegressor

SEED = 42

In [4]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import (
    regression_metrics,
    relative_improvement
)

from src.evaluation.calibration import (
    calibration_by_quantile
)

## 3. Carga de datos

In [5]:
PORTFOLIO_PATH = Path(
    "../data/raw/synthetic_insurance_portfolio.csv"
)

CLAIMS_PATH = Path(
    "../data/raw/synthetic_insurance_claims.csv"
)

SPLIT_PATH = Path(
    "../data/processed/train_test_split.csv"
)

df = pd.read_csv(PORTFOLIO_PATH)
df_claims = pd.read_csv(CLAIMS_PATH)
split_df = pd.read_csv(SPLIT_PATH)

In [ ]:
# Aplicamos el split ya definido
df = df.merge(
    split_df,
    on="policy_id",
    how="left"
)

df_train = df.loc[
    df["split"] == "train"
].copy()

df_test = df.loc[
    df["split"] == "test"
].copy()

In [7]:
df_claims = df_claims.merge(
    split_df,
    on="policy_id",
    how="left"
)

claims_train = df_claims.loc[
    df_claims["split"] == "train"
].copy()

claims_test = df_claims.loc[
    df_claims["split"] == "test"
].copy()

In [ ]:
# Checks
assert df["split"].notna().all()
assert df_claims["split"].notna().all()

print(
    f"Pólizas train/test: "
    f"{len(df_train):,} / {len(df_test):,}"
)

print(
    f"Siniestros train/test: "
    f"{len(claims_train):,} / {len(claims_test):,}"
)

Pólizas train/test: 8,000 / 2,000
Siniestros train/test: 1,239 / 315


## 4. Preprocesamiento.

In [9]:
NUMERIC_FEATURES = [
    "edad",
    "anios_vehiculo",
    "historial_siniestros"
]

CATEGORICAL_FEATURES = [
    "zona",
    "cobertura",
    "tipo_uso"
]

FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="passthrough"
)

## 5. Frecuencia

### 5.1 Target

In [11]:
X_train_freq = df_train[FEATURES]
X_test_freq = df_test[FEATURES]

y_train_freq = df_train["numero_siniestros"]
y_test_freq = df_test["numero_siniestros"]

## 6. Random Forest para frecuencia

In [12]:
rf_frequency = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=5,
                random_state=SEED,
                n_jobs=-1,
                criterion="poisson"  # usar una reducción de devianza Poisson en lugar de squared error.
            )
        )
    ]
)

### 6.1 Medición del tiempo de entrenamiento

In [13]:
start = perf_counter()

rf_frequency.fit(
    X_train_freq,
    y_train_freq
)

rf_frequency_train_time = (
    perf_counter() - start
)

print(
    f"Tiempo de entrenamiento RF: "
    f"{rf_frequency_train_time:.3f} segundos"
)

Tiempo de entrenamiento RF: 0.619 segundos


### 6.2 Predicciones

In [14]:
pred_train_freq_rf = rf_frequency.predict(
    X_train_freq
)

pred_test_freq_rf = rf_frequency.predict(
    X_test_freq
)

In [ ]:
# Comprobar que no hay predicciones negativas ya que estamos usando criterio Poisson y targets no negativos.
print(
    pred_test_freq_rf.min(),
    pred_test_freq_rf.max()
)

0.015847263513007605 0.7521111385351374


### 6.3 Evaluación inicial del Random Forest de frecuencia

El Random Forest no mejora el desempeño out-of-sample del GLM Poisson.

El modelo obtiene un MAE de 0.2578, un RMSE de 0.3978 y una Poisson deviance
de 0.6224. Estas métricas son ligeramente peores que las obtenidas previamente
por el GLM Poisson.

Por tanto, el incremento de flexibilidad del Random Forest no se traduce,
en esta primera especificación, en una mejora predictiva sobre los conteos
observados.

In [16]:
baseline_freq = np.full(
    len(y_test_freq),
    y_train_freq.mean()
)

In [17]:
rf_freq_mae = mean_absolute_error(
    y_test_freq,
    pred_test_freq_rf
)

rf_freq_rmse = np.sqrt(
    mean_squared_error(
        y_test_freq,
        pred_test_freq_rf
    )
)

rf_freq_deviance = mean_poisson_deviance(
    y_test_freq,
    pred_test_freq_rf
)

print(f"MAE: {rf_freq_mae:.4f}")
print(f"RMSE: {rf_freq_rmse:.4f}")
print(
    f"Poisson deviance: "
    f"{rf_freq_deviance:.4f}"
)

MAE: 0.2578
RMSE: 0.3978
Poisson deviance: 0.6224


### 6.4 Evaluación contra la frecuencia verdadera (*oracle*)

La evaluación contra `lambda_real` revela una diferencia mucho más importante
entre Random Forest y GLM.

El Random Forest obtiene un MAE oracle de 0.0603, un RMSE de 0.0816 y una
correlación de 0.6865 con la frecuencia verdadera.

Estos resultados son considerablemente peores que los obtenidos mediante el
GLM Poisson, cuya correlación oracle era aproximadamente 0.98.

Esto muestra que métricas calculadas únicamente contra los conteos observados
pueden ocultar diferencias importantes entre modelos debido a la elevada
variabilidad aleatoria del proceso Poisson.

En este experimento, el Random Forest está ajustando parcialmente el ruido de
las realizaciones individuales en lugar de recuperar con precisión la función
de frecuencia esperada subyacente.

In [18]:
lambda_true_test = df_test[
    "lambda_real"
]

rf_freq_oracle_mae = (
    mean_absolute_error(
        lambda_true_test,
        pred_test_freq_rf
    )
)

rf_freq_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_rf
    )
)

rf_freq_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_rf
)[0, 1]

print(
    f"Oracle MAE: "
    f"{rf_freq_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{rf_freq_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{rf_freq_oracle_corr:.4f}"
)

Oracle MAE: 0.060336
Oracle RMSE: 0.081628
Oracle correlation: 0.6865


### 6.5 Train vs Test: revisar que no haya overfitting

El desempeño del Random Forest es considerablemente mejor en entrenamiento
que en prueba.

El MAE aumenta de aproximadamente 0.215 en train a 0.258 en test, mientras
que el RMSE pasa de 0.334 a 0.398.

Esta diferencia sugiere que el modelo está capturando parte de la variabilidad
específica de la muestra de entrenamiento y presenta cierto grado de
sobreajuste.

Por este motivo, antes de descartar Random Forest resulta conveniente estudiar
una especificación más regularizada.


In [19]:
rf_train_metrics = regression_metrics(
    y_train_freq,
    pred_train_freq_rf
)

rf_test_metrics = regression_metrics(
    y_test_freq,
    pred_test_freq_rf
)

pd.DataFrame(
    [
        rf_train_metrics,
        rf_test_metrics
    ],
    index=["Train", "Test"]
)

,MAE,RMSE
Train,0.214757,0.333952
Test,0.257800,0.397768


### 6.6 Ajuste de hiperparámetros del Random Forest

El primer Random Forest presentó una diferencia apreciable entre el desempeño
de entrenamiento y prueba, además de una recuperación considerablemente peor
de la frecuencia verdadera que el GLM Poisson.

Antes de comparar definitivamente ambos enfoques, se realizará una búsqueda
moderada de hiperparámetros orientada principalmente a controlar la complejidad
del bosque.

La selección se realizará exclusivamente mediante validación cruzada dentro de
la muestra de entrenamiento. El conjunto de prueba y las cantidades *oracle*
no participan en ninguna decisión de modelado.

#### 6.6.1 Espacio de busqueda

In [ ]:
param_distributions = {
    "model__max_depth": [           # Qué tan complejos pueden ser los árboles (profundidad)
        4,
        6,
        8,
        10,
        None
    ],
    "model__min_samples_leaf": [    # Impide generar hojas sustentadas por muy pocas pólizas
        5,
        10,
        20,
        40
    ],
    "model__max_features": [        # Controla la cantidad de variables que puede considerar cada árbol en cada
        "sqrt",                     # división
        0.7,
        1.0
    ]
}

#### 6.6.2 Scorer de Poisson deviance

In [24]:
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_poisson_deviance


def poisson_deviance_score(y_true, y_pred):
    y_pred = np.clip(
        y_pred,
        1e-10,
        None
    )

    return mean_poisson_deviance(
        y_true,
        y_pred
    )


poisson_scorer = make_scorer(
    poisson_deviance_score,
    greater_is_better=False
)

#### 6.6.3 RandomizedSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV


rf_search = RandomizedSearchCV(
    estimator=rf_frequency,
    param_distributions=param_distributions,
    n_iter=20,
    scoring=poisson_scorer,
    cv=5,
    refit=True,                     # Entrena el mejor modelo sobre todo df_train después de elegirla
    return_train_score=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

In [26]:
start = perf_counter()

rf_search.fit(
    X_train_freq,
    y_train_freq
)

rf_search_time = (
    perf_counter() - start
)

print(
    f"Tiempo búsqueda: "
    f"{rf_search_time:.2f} segundos"
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Tiempo búsqueda: 29.08 segundos


#### 6.6.4 Mejor configuración

In [27]:
print(
    "Mejores parámetros:"
)

print(
    rf_search.best_params_
)

print(
    "\nPoisson deviance CV:"
)

print(
    -rf_search.best_score_
)

Mejores parámetros:
{'model__min_samples_leaf': 20, 'model__max_features': 0.7, 'model__max_depth': 4}

Poisson deviance CV:
0.5853769943876005


In [28]:
rf_frequency_tuned = (
    rf_search.best_estimator_
)

#### 6.6.5 Mejores configuraciones

La búsqueda aleatoria seleccionó un Random Forest relativamente poco profundo,
con `max_depth=4`, `min_samples_leaf=20` y `max_features=0.7`.

Esta configuración reduce considerablemente la capacidad del modelo respecto
al bosque inicial.

In [29]:
cv_results = pd.DataFrame(
    rf_search.cv_results_
)

rf_cv_summary = pd.DataFrame({
    "rank": cv_results[
        "rank_test_score"
    ],

    "mean_train_deviance": (
        -cv_results[
            "mean_train_score"
        ]
    ),

    "mean_validation_deviance": (
        -cv_results[
            "mean_test_score"
        ]
    ),

    "std_validation": (
        cv_results[
            "std_test_score"
        ]
    ),

    "max_depth": cv_results[
        "param_model__max_depth"
    ],

    "min_samples_leaf": cv_results[
        "param_model__min_samples_leaf"
    ],

    "max_features": cv_results[
        "param_model__max_features"
    ]
})

rf_cv_summary = (
    rf_cv_summary
    .sort_values("rank")
    .head(10)
)

rf_cv_summary

,rank,mean_train_deviance,mean_validation_deviance,std_validation,max_depth,min_samples_leaf,max_features
17,1,0.567979,0.585377,0.006970,4,20,0.7
1,2,0.567341,0.585619,0.006946,4,10,0.7
19,3,0.566908,0.585684,0.006972,4,5,0.7
11,4,0.524415,0.586046,0.008886,None,20,sqrt
4,5,0.555706,0.586369,0.008125,6,10,sqrt
8,6,0.552908,0.586549,0.008105,6,5,sqrt
12,7,0.539008,0.587072,0.008604,8,40,0.7
15,8,0.539985,0.587572,0.007212,6,10,0.7
16,9,0.565140,0.587586,0.006705,4,5,1.0
0,10,0.576406,0.588435,0.008244,4,5,sqrt


### 6.7 Evaluación sobre test

La regularización elimina gran parte de la diferencia entre entrenamiento y
prueba: el RMSE pasa de aproximadamente 0.389 en train a 0.392 en test.

In [30]:
pred_train_freq_rf_tuned = (
    rf_frequency_tuned.predict(
        X_train_freq
    )
)

pred_test_freq_rf_tuned = (
    rf_frequency_tuned.predict(
        X_test_freq
    )
)

In [31]:
rf_tuned_train_metrics = (
    regression_metrics(
        y_train_freq,
        pred_train_freq_rf_tuned
    )
)

rf_tuned_test_metrics = (
    regression_metrics(
        y_test_freq,
        pred_test_freq_rf_tuned
    )
)

pd.DataFrame(
    [
        rf_tuned_train_metrics,
        rf_tuned_test_metrics
    ],
    index=[
        "Train",
        "Test"
    ]
)

,MAE,RMSE
Train,0.255268,0.389103
Test,0.258924,0.392079


In [32]:
rf_tuned_deviance = (
    mean_poisson_deviance(
        y_test_freq,
        np.clip(
            pred_test_freq_rf_tuned,
            1e-10,
            None
        )
    )
)

print(
    f"Poisson deviance test: "
    f"{rf_tuned_deviance:.4f}"
)

Poisson deviance test: 0.5925


La recuperación de la frecuencia verdadera mejora de manera importante.
La correlación con `lambda_real` aumenta de aproximadamente 0.69 a 0.88 y el
RMSE oracle disminuye de 0.0816 a 0.0375.

Esto sugiere que el Random Forest inicial estaba capturando parcialmente
variabilidad aleatoria de los conteos observados en lugar de la estructura
subyacente de frecuencia.

In [33]:
rf_tuned_oracle_mae = (
    mean_absolute_error(
        lambda_true_test,
        pred_test_freq_rf_tuned
    )
)

rf_tuned_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_rf_tuned
    )
)

rf_tuned_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_rf_tuned
)[0, 1]


print(
    f"Oracle MAE: "
    f"{rf_tuned_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{rf_tuned_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{rf_tuned_oracle_corr:.4f}"
)

Oracle MAE: 0.026815
Oracle RMSE: 0.037533
Oracle correlation: 0.8845


### 6.8 Comparación final: GLM Poisson vs Random Forest

La regularización mejora considerablemente el desempeño del Random Forest,
especialmente en términos de Poisson deviance y recuperación de la frecuencia
verdadera.

Sin embargo, el GLM Poisson continúa mostrando mejores resultados.

Sobre los conteos observados, las diferencias son relativamente pequeñas:
el GLM obtiene menor MAE, RMSE y Poisson deviance.

La diferencia resulta mucho más clara al utilizar el oracle. El GLM alcanza
una correlación de aproximadamente 0.98 con la frecuencia verdadera y un RMSE
oracle de 0.0179, frente a 0.88 y 0.0375 para el Random Forest ajustado.

Por tanto, aunque la regularización permite que Random Forest generalice mucho
mejor, la flexibilidad adicional del modelo no proporciona una ventaja frente
a un GLM cuya estructura funcional se encuentra bien alineada con el problema.

In [34]:
frequency_comparison = pd.DataFrame({
    "Modelo": [
        "GLM Poisson",
        "Random Forest inicial",
        "Random Forest tuned"
    ],

    "MAE test": [
        0.255862,
        rf_freq_mae,
        rf_tuned_test_metrics["MAE"]
    ],

    "RMSE test": [
        0.390971,
        rf_freq_rmse,
        rf_tuned_test_metrics["RMSE"]
    ],

    "Poisson deviance": [
        0.5872,
        rf_freq_deviance,
        rf_tuned_deviance
    ],

    "Oracle MAE": [
        0.013578,
        rf_freq_oracle_mae,
        rf_tuned_oracle_mae
    ],

    "Oracle RMSE": [
        0.017865,
        rf_freq_oracle_rmse,
        rf_tuned_oracle_rmse
    ],

    "Oracle correlation": [
        0.9814,
        rf_freq_oracle_corr,
        rf_tuned_oracle_corr
    ]
})

frequency_comparison

,Modelo,MAE test,RMSE test,Poisson deviance,Oracle MAE,Oracle RMSE,Oracle correlation
0,GLM Poisson,0.255862,0.390971,0.587200,0.013578,0.017865,0.981400
1,Random Forest inicial,0.257800,0.397768,0.622380,0.060336,0.081628,0.686451
2,Random Forest tuned,0.258924,0.392079,0.592515,0.026815,0.037533,0.884533


## 7. XGBoost para frecuencia

Como segundo modelo de Machine Learning se utilizará Gradient Boosting mediante
XGBoost.

A diferencia de Random Forest, donde los árboles se construyen de manera
aproximadamente independiente y posteriormente se promedian, boosting genera
árboles secuencialmente.

Cada nuevo árbol intenta mejorar los errores cometidos por el conjunto de
árboles anteriores.

Para el problema de frecuencia se utiliza una función objetivo Poisson, de
manera que el modelo permanezca alineado con la naturaleza no negativa y de
conteo del target.

### 7.1 Modelo inicial

In [ ]:
xgb_frequency = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            XGBRegressor(
                objective="count:poisson",
                n_estimators=400,
                learning_rate=0.05,     # Hace que cada árbol haga una contribución relativamente pequeña.
                max_depth=3,            # Limita interacciones muy complejas.
                min_child_weight=10,    # Dificulta crear ramas basadas en muy poca información.
                subsample=0.8,          # subsample y colsample_bytree añaden aleatoriedad
                colsample_bytree=0.8,   # para ayudar con generalizacion.
                reg_lambda=1.0,
                random_state=SEED,
                n_jobs=-1,
                tree_method="hist"
            )
        )
    ]
)

### 7.2 Entrenamiento y tiempo

In [36]:
start = perf_counter()

xgb_frequency.fit(
    X_train_freq,
    y_train_freq
)

xgb_frequency_train_time = (
    perf_counter() - start
)

print(
    f"Tiempo entrenamiento XGBoost: "
    f"{xgb_frequency_train_time:.3f} segundos"
)

Tiempo entrenamiento XGBoost: 0.382 segundos


### 7.3 Predicciones

In [37]:
pred_train_freq_xgb = (
    xgb_frequency.predict(
        X_train_freq
    )
)

pred_test_freq_xgb = (
    xgb_frequency.predict(
        X_test_freq
    )
)

In [ ]:
# Con count:poisson deberían ser estrictamente positivas.
print(
    pred_test_freq_xgb.min(),
    pred_test_freq_xgb.max()
)

0.033658996 0.7760566


### 7.4 Métricas observables

El modelo XGBoost presenta un desempeño competitivo respecto al Random Forest
regularizado, aunque todavía no supera al GLM Poisson.

En test obtiene un MAE de 0.2565, un RMSE de 0.3926 y una Poisson deviance de
0.5928. Estas métricas se encuentran muy próximas a las del Random Forest
ajustado y ligeramente por detrás del GLM.

La diferencia entre entrenamiento y prueba es moderada, por lo que no se
observa el sobreajuste pronunciado presentado por el Random Forest inicial.

In [39]:
xgb_freq_train_metrics = regression_metrics(
    y_train_freq,
    pred_train_freq_xgb
)

xgb_freq_test_metrics = regression_metrics(
    y_test_freq,
    pred_test_freq_xgb
)

pd.DataFrame(
    [
        xgb_freq_train_metrics,
        xgb_freq_test_metrics
    ],
    index=[
        "Train",
        "Test"
    ]
)

,MAE,RMSE
Train,0.248632,0.382990
Test,0.256489,0.392573


In [40]:
xgb_freq_deviance = mean_poisson_deviance(
    y_test_freq,
    pred_test_freq_xgb
)

print(
    f"Poisson deviance test: "
    f"{xgb_freq_deviance:.6f}"
)

Poisson deviance test: 0.592776


### 7.5 Evaluación Oracle

En la evaluación oracle, XGBoost alcanza una correlación de 0.91 con la
frecuencia verdadera, superior a la obtenida por Random Forest. Sin embargo,
sus errores absolutos frente al oracle permanecen considerablemente por encima
de los del GLM Poisson.

Esto sugiere que XGBoost recupera razonablemente bien la estructura relativa
del riesgo, aunque todavía presenta errores en la magnitud de la frecuencia
esperada.

In [41]:
xgb_freq_oracle_mae = mean_absolute_error(
    lambda_true_test,
    pred_test_freq_xgb
)

xgb_freq_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_xgb
    )
)

xgb_freq_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_xgb
)[0, 1]

print(
    f"Oracle MAE: "
    f"{xgb_freq_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{xgb_freq_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{xgb_freq_oracle_corr:.4f}"
)

Oracle MAE: 0.027258
Oracle RMSE: 0.038105
Oracle correlation: 0.9102


In [42]:
frequency_comparison_xgb = pd.DataFrame({
    "Modelo": [
        "GLM Poisson",
        "Random Forest tuned",
        "XGBoost inicial"
    ],

    "MAE test": [
        0.255862,
        rf_tuned_test_metrics["MAE"],
        xgb_freq_test_metrics["MAE"]
    ],

    "RMSE test": [
        0.390971,
        rf_tuned_test_metrics["RMSE"],
        xgb_freq_test_metrics["RMSE"]
    ],

    "Poisson deviance": [
        0.5872,
        rf_tuned_deviance,
        xgb_freq_deviance
    ],

    "Oracle MAE": [
        0.013578,
        rf_tuned_oracle_mae,
        xgb_freq_oracle_mae
    ],

    "Oracle RMSE": [
        0.017865,
        rf_tuned_oracle_rmse,
        xgb_freq_oracle_rmse
    ],

    "Oracle correlation": [
        0.9814,
        rf_tuned_oracle_corr,
        xgb_freq_oracle_corr
    ]
})

frequency_comparison_xgb

,Modelo,MAE test,RMSE test,Poisson deviance,Oracle MAE,Oracle RMSE,Oracle correlation
0,GLM Poisson,0.255862,0.390971,0.587200,0.013578,0.017865,0.981400
1,Random Forest tuned,0.258924,0.392079,0.592515,0.026815,0.037533,0.884533
2,XGBoost inicial,0.256489,0.392573,0.592776,0.027258,0.038105,0.910216


### 7.6 Ajuste de hiperpárametros

In [43]:
xgb_param_distributions = {
    "model__n_estimators": [
        200, 400, 600, 800
    ],

    "model__learning_rate": [
        0.02, 0.05, 0.08, 0.10
    ],

    "model__max_depth": [
        2, 3, 4, 5
    ],

    "model__min_child_weight": [
        1, 5, 10, 20
    ],

    "model__subsample": [
        0.7, 0.8, 1.0
    ],

    "model__colsample_bytree": [
        0.7, 0.8, 1.0
    ],

    "model__reg_lambda": [
        1.0, 5.0, 10.0
    ],

    "model__reg_alpha": [
        0.0, 0.1, 0.5
    ]
}

#### 7.6.1 Espacio de busqueda

In [ ]:

xgb_search = RandomizedSearchCV(
    estimator=xgb_frequency,
    param_distributions=xgb_param_distributions,
    n_iter=25,
    scoring=poisson_scorer,
    cv=5,
    refit=True,
    return_train_score=True,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

#### 7.6.2 Mejor modelo

In [45]:
start = perf_counter()

xgb_search.fit(
    X_train_freq,
    y_train_freq
)

xgb_search_time = perf_counter() - start

print(
    f"Tiempo búsqueda XGBoost: "
    f"{xgb_search_time:.2f} segundos"
)

print("\nMejores parámetros:")
print(xgb_search.best_params_)

print("\nPoisson deviance CV:")
print(-xgb_search.best_score_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Tiempo búsqueda XGBoost: 15.90 segundos

Mejores parámetros:
{'model__subsample': 1.0, 'model__reg_lambda': 10.0, 'model__reg_alpha': 0.5, 'model__n_estimators': 200, 'model__min_child_weight': 1, 'model__max_depth': 2, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}

Poisson deviance CV:
0.5837377429008483


In [46]:
xgb_cv_results = pd.DataFrame(
    xgb_search.cv_results_
)

xgb_cv_summary = pd.DataFrame({
    "rank":
        xgb_cv_results["rank_test_score"],

    "train_deviance":
        -xgb_cv_results["mean_train_score"],

    "validation_deviance":
        -xgb_cv_results["mean_test_score"],

    "std_validation":
        xgb_cv_results["std_test_score"],

    "n_estimators":
        xgb_cv_results[
            "param_model__n_estimators"
        ],

    "learning_rate":
        xgb_cv_results[
            "param_model__learning_rate"
        ],

    "max_depth":
        xgb_cv_results[
            "param_model__max_depth"
        ],

    "min_child_weight":
        xgb_cv_results[
            "param_model__min_child_weight"
        ],

    "subsample":
        xgb_cv_results[
            "param_model__subsample"
        ],

    "colsample":
        xgb_cv_results[
            "param_model__colsample_bytree"
        ],

    "reg_lambda":
        xgb_cv_results[
            "param_model__reg_lambda"
        ],

    "reg_alpha":
        xgb_cv_results[
            "param_model__reg_alpha"
        ]
})

xgb_cv_summary = (
    xgb_cv_summary
    .sort_values("rank")
    .head(10)
)

xgb_cv_summary

,rank,train_deviance,validation_deviance,std_validation,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample,reg_lambda,reg_alpha
21,1,0.571148,0.583738,0.007847,200,0.05,2,1,1.0,0.8,10.0,0.5
15,2,0.563192,0.584788,0.008896,200,0.10,2,5,0.8,0.7,1.0,0.5
13,3,0.573033,0.586063,0.008079,200,0.02,3,20,0.8,0.7,5.0,0.1
0,4,0.561230,0.586178,0.008751,400,0.05,2,10,0.7,1.0,1.0,0.0
11,5,0.549943,0.587269,0.007472,200,0.02,5,5,0.7,0.8,10.0,0.0
1,6,0.554608,0.589157,0.008316,800,0.02,3,20,1.0,0.7,5.0,0.5
8,7,0.554157,0.589167,0.008639,600,0.02,3,5,1.0,1.0,5.0,0.0
19,8,0.545770,0.592080,0.008556,200,0.10,3,20,1.0,1.0,5.0,0.0
10,9,0.539532,0.592622,0.009487,400,0.05,3,1,0.7,1.0,5.0,0.1
18,10,0.551537,0.593448,0.008177,600,0.08,2,10,0.7,1.0,1.0,0.5


##### 7.6.3 Evaluación final

In [47]:
xgb_frequency_tuned = (
    xgb_search.best_estimator_
)

pred_train_freq_xgb_tuned = (
    xgb_frequency_tuned.predict(
        X_train_freq
    )
)

pred_test_freq_xgb_tuned = (
    xgb_frequency_tuned.predict(
        X_test_freq
    )
)

In [48]:
xgb_tuned_train_metrics = (
    regression_metrics(
        y_train_freq,
        pred_train_freq_xgb_tuned
    )
)

xgb_tuned_test_metrics = (
    regression_metrics(
        y_test_freq,
        pred_test_freq_xgb_tuned
    )
)

pd.DataFrame(
    [
        xgb_tuned_train_metrics,
        xgb_tuned_test_metrics
    ],
    index=[
        "Train",
        "Test"
    ]
)

,MAE,RMSE
Train,0.255527,0.389761
Test,0.258263,0.391349


In [49]:
xgb_tuned_deviance = (
    mean_poisson_deviance(
        y_test_freq,
        pred_test_freq_xgb_tuned
    )
)

print(
    f"Poisson deviance test: "
    f"{xgb_tuned_deviance:.6f}"
)

Poisson deviance test: 0.589806


In [50]:
xgb_tuned_oracle_mae = (
    mean_absolute_error(
        lambda_true_test,
        pred_test_freq_xgb_tuned
    )
)

xgb_tuned_oracle_rmse = np.sqrt(
    mean_squared_error(
        lambda_true_test,
        pred_test_freq_xgb_tuned
    )
)

xgb_tuned_oracle_corr = np.corrcoef(
    lambda_true_test,
    pred_test_freq_xgb_tuned
)[0, 1]

print(
    f"Oracle MAE: "
    f"{xgb_tuned_oracle_mae:.6f}"
)

print(
    f"Oracle RMSE: "
    f"{xgb_tuned_oracle_rmse:.6f}"
)

print(
    f"Oracle correlation: "
    f"{xgb_tuned_oracle_corr:.4f}"
)

Oracle MAE: 0.022880
Oracle RMSE: 0.031772
Oracle correlation: 0.9212
